### Import Dependencies

In [1]:
import random
from qdrant_client import QdrantClient
import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
import numpy as np
from qdrant_client import QdrantClient

### Fictional Warehouses

In [2]:
warehouses = [
    {
        "warehouse_id": "DE-BER-01",
        "warehouse_location": "Berlin, Germany",
        "warehouse_name": "Berlin Distribution Center"
    },
    {
        "warehouse_id": "DE-MUN-01",
        "warehouse_location": "Munich, Germany",
        "warehouse_name": "Munich Logistics Hub"
    },
    {
        "warehouse_id": "DE-HAM-01",
        "warehouse_location": "Hamburg, Germany",
        "warehouse_name": "Hamburg North Warehouse"
    },
    {
        "warehouse_id": "FR-PAR-01",
        "warehouse_location": "Paris, France",
        "warehouse_name": "Paris Central Depot"
    },
    {
        "warehouse_id": "FR-LYO-01",
        "warehouse_location": "Lyon, France",
        "warehouse_name": "Lyon Regional Warehouse"
    },
    {
        "warehouse_id": "FR-MAR-01",
        "warehouse_location": "Marseille, France",
        "warehouse_name": "Marseille Mediterranean Hub"
    }
]

### Simulate Stock Availability for each of the warehouses

#### Retrieve all item IDs from the Amamzon items Qdrant Collection

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
dummy_vector = np.zeros(1536).tolist()

In [5]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
).points

In [6]:
payload

[ScoredPoint(id=181, version=20, score=0.0, payload={'parent_asin': 'B0BVLZ4R8W'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=78, version=20, score=0.0, payload={'parent_asin': 'B09ZQ7VBGG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=149, version=20, score=0.0, payload={'parent_asin': 'B09TV65GD2'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=203, version=20, score=0.0, payload={'parent_asin': 'B09R389FRD'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=215, version=20, score=0.0, payload={'parent_asin': 'B0B3NCVBPK'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=16, version=20, score=0.0, payload={'parent_asin': 'B0B3M8PDRX'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=96, version=20, score=0.0, payload={'parent_asin': 'B09QH6G31R'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=461, version=20, score=0.0, payload={'parent_asin': 'B0B1TPKCPB'}, vec

In [7]:
len(payload)

1000

In [8]:
parent_asin_list = [item.payload["parent_asin"] for item in payload]

In [9]:
parent_asin_list

['B0BVLZ4R8W',
 'B09ZQ7VBGG',
 'B09TV65GD2',
 'B09R389FRD',
 'B0B3NCVBPK',
 'B0B3M8PDRX',
 'B09QH6G31R',
 'B0B1TPKCPB',
 'B0BVDX1HQ9',
 'B09X6GGCNQ',
 'B09ZLKRKL7',
 'B0B1DW3VP6',
 'B0BSXMFV7F',
 'B09Q97PB4G',
 'B0BMQKP9Y2',
 'B0BMQLQMWJ',
 'B09WB36XNT',
 'B09SP22GTK',
 'B0C61QHRB6',
 'B09T614FYF',
 'B09ZLQW9ZT',
 'B0BQCMT1VR',
 'B0BS1N1X15',
 'B0BRSZDPFF',
 'B0B7F9JN5J',
 'B09P4CT5F9',
 'B0B3DLRRTX',
 'B0BTTKQT4S',
 'B0BHGWCV4P',
 'B00BSWMIOK',
 'B09PVSQMZS',
 'B09SV8DDH1',
 'B09VTVK4JY',
 'B09VTX1NC5',
 'B09V3868MP',
 'B09XX899VW',
 'B0B51HHL23',
 'B0B46P6H1B',
 'B09RGPZCRX',
 'B09SP1GQYV',
 'B0B287C758',
 'B09X1RR31C',
 'B0B31XM1JC',
 'B0B6YDH5V8',
 'B0BL8VCTKK',
 'B09QWG1HLQ',
 'B0B6GJYWXQ',
 'B0BTW59YJJ',
 'B09VCS9Q5W',
 'B0BLWGL217',
 'B0B142DPG8',
 'B0BP4R3GMB',
 'B09QNYKHW4',
 'B09R3BYZ5N',
 'B0B2WMXB6C',
 'B00008EN57',
 'B09PRMQP5C',
 'B0B4PJ64KC',
 'B0BY32TVLL',
 'B0BPCZ1N6K',
 'B09VDM34YS',
 'B0B3DNW3HF',
 'B07R826VJT',
 'B0BDSXJ65P',
 'B0BHGBB3QH',
 'B09VL6W7PQ',
 'B0BGCD67

### Generate Synthetic availability for all items in Qdrant

In [10]:
def generate_inventory_data(warehouses, product_ids, availability_rate=0.75):
    
    inventory_records = []
    
    for warehouse in warehouses:
        for product_id in product_ids:
            # 75% chance the product is available in this warehouse
            if random.random() < availability_rate:
                total_quantity = random.randint(0, 100)
                
                # Only add to inventory if quantity > 0
                if total_quantity > 0:
                    inventory_records.append({
                        "warehouse_id": warehouse["warehouse_id"],
                        "warehouse_location": warehouse["warehouse_location"],
                        "warehouse_name": warehouse["warehouse_name"],
                        "product_id": product_id,
                        "total_quantity": total_quantity,
                        "reserved_quantity": 0  # Starting with no reservations
                    })
    
    return inventory_records

In [11]:
inventory_data = generate_inventory_data(warehouses, parent_asin_list)

In [12]:
inventory_data

[{'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0BVLZ4R8W',
  'total_quantity': 37,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09TV65GD2',
  'total_quantity': 19,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09R389FRD',
  'total_quantity': 94,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0B3NCVBPK',
  'total_quantity': 5,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0B3M8PDRX',
  'total_quantity': 80,
  '

In [13]:
1000*0.75*6*0.99

4455.0

In [14]:
len(inventory_data)

4476

### Write synthetic data to Postgres

In [15]:
def insert_inventory_to_db(inventory_records):
   
    try:
        # Connect to the database
        conn = psycopg2.connect(
            host="localhost",
            port=5433,
            database="tools_database",
            user="tools_user",
            password="tools_user_password"
        )
        conn.autocommit = True

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
            # Prepare the INSERT query
            insert_query = """
            INSERT INTO warehouses.inventory 
            (warehouse_id, warehouse_location, warehouse_name, product_id, total_quantity, reserved_quantity)
            VALUES (%(warehouse_id)s, %(warehouse_location)s, %(warehouse_name)s, %(product_id)s, %(total_quantity)s, %(reserved_quantity)s)
            """
            
            # Use execute_batch for better performance with many inserts
            execute_batch(cursor, insert_query, inventory_records, page_size=100)
            
            # Commit the transaction
            conn.commit()
            
            print(f"Successfully inserted {len(inventory_records)} records into warehouses.inventory")
            
            # Close cursor and connection
            cursor.close()
            conn.close()
        
    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

In [16]:
insert_inventory_to_db(inventory_data)

Database error: duplicate key value violates unique constraint "unique_warehouse_product"
DETAIL:  Key (warehouse_id, product_id)=(DE-BER-01, B0B82SL4NC) already exists.

